# Dadi-Cli

In [8]:
from preprocess import Processor
from demographic_models import bottleneck_model
import msprime

In [14]:
upper_bound_params = {
    "N0": 10000,
    "Nb": 5000,
    "N_recover": 7000,
    "t_bottleneck_end": 1000,
    "t_bottleneck_start": 2000,
}
lower_bound_params = {
    "N0": 8000,
    "Nb": 4000,
    "N_recover": 6000,
    "t_bottleneck_end": 800,
    "t_bottleneck_start": 1500,
}
model_config = {
    "input_size": 10,
    "hidden_size": 1000,
    "output_size": 5,
    "num_epochs": 1000,
    "learning_rate": 3e-4,
    "num_layers": 1,
    "dropout_rate": 0.1,
    "weight_decay": 1e-4,
    "parameter_names": ["N0", "Nb", "N_recover", "t_bottleneck_end", "t_bottleneck_start"], # these should be a list of parameters that we want to optimize 

}

config = {
    "upper_bound_params": upper_bound_params,
    "lower_bound_params": lower_bound_params,
    "num_sims_pretrain": 100,
    "num_sims_inference": 100,
    "num_samples": 20,
    "experiment_name": "dadi_moments_analysis_new_analysis_optimize_regularize_1layers",
    "dadi_analysis": True,
    "moments_analysis": True,
    "momentsLD_analysis": False,
    "num_windows": 50,
    "window_length": 1e4,
    "maxiter": 100,
    "genome_length": 1e8,
    "mutation_rate": 1.26e-8,
    "recombination_rate": 1.007e-8,
    "seed": 42,
    "normalization": False,
    "remove_outliers": True,
    "use_FIM": False,
    "neural_net_hyperparameters": model_config,
    "demographic_model": "bottleneck_model",
    "parameter_names": ["N0", "Nb", "N_recover", "t_bottleneck_end", "t_bottleneck_start"], # these should be a list of parameters that we want to optimize 
    "optimization_initial_guess": [0.25, 0.75, 0.1, 0.05],
    "vcf_filepath": "/sietch_colab/akapoor/GHIST-bottleneck.vcf.gz",
    "txt_filepath": "/sietch_colab/akapoor/wisent.txt",
    "popname": "wisent"
    
}

In [15]:
processor = Processor(config, '/sietch_colab/akapoor/')


In [16]:
demographic_model = split_isolation_model

In [17]:
sampled_params = processor.sample_params()

In [18]:
g = demographic_model(sampled_params)
    
demog = msprime.Demography.from_demes(g)
ts = msprime.sim_ancestry(
    {"A": processor.num_samples},
    demography=demog,
    sequence_length=processor.L,
    recombination_rate=processor.recombination_rate,
    random_seed=295,
)
ts = msprime.sim_mutations(ts, rate=config['mutation_rate'])